# Phase 0: Creating the CTL_GLYCO_GENES_WIDE Table

Extract glycogene expression data from CMAP Connectivity Map 2020 (ctl_predicted_RNAseq_profiles.gctx) and upload to Snowflake.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import os
import logging
from typing import List, Optional, Dict
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import serialization
from snowflake.connector import connect
from cmapPy.pandasGEXpress.parse import parse

# Logging configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Add project root to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

logger.info(f"Project root: {project_root}")

## Data File Path Configuration

Configure paths for CTL predicted RNA-seq profiles and glycogene list.

In [ ]:
# Data file paths
data_dir = project_root / "sample" / "data"
glycogenes_list_path = project_root / "GlycoEnzOnto" / "glycogenes_from_gmt.txt"

# CMAP Connectivity Map 2020 - CTL predicted RNA-seq profiles
gctx_file = data_dir / "ctl_predicted_RNAseq_profiles.gctx"

# Check file existence
files_to_check = {
    "gctx": gctx_file,
    "glycogenes_list": glycogenes_list_path
}

logger.info("Checking files:")
for name, path in files_to_check.items():
    if path.exists():
        size_gb = path.stat().st_size / (1024**3)
        logger.info(f"✓ {name}: {path} ({size_gb:.2f} GB)")
    else:
        logger.error(f"✗ {name} not found: {path}")
        raise FileNotFoundError(f"{name} not found: {path}")

## Loading the Glycogene List

Load the glycogene list obtained from GlycoEnzOnto.

In [ ]:
# Load glycogene list
with open(glycogenes_list_path, 'r') as f:
    glycogenes_raw = [line.strip().strip('"') for line in f if line.strip()]

# Remove duplicates and sort
glycogenes = sorted(list(set(glycogenes_raw)))

logger.info(f"Number of glycogenes: {len(glycogenes)}")
logger.info(f"First 10 genes: {glycogenes[:10]}")
logger.info(f"Last 10 genes: {glycogenes[-10:]}")

## Loading the GCTX File

Load the CTL predicted RNA-seq profiles (gctx format) using the cmapPy library.

In [ ]:
logger.info("Loading CTL predicted RNA-seq profiles GCTX file...")
logger.info(f"File size: {gctx_file.stat().st_size / (1024**3):.2f} GB")

# Load gctx file using cmapPy
gctoo = parse(str(gctx_file))

logger.info(f"Data shape: {gctoo.data_df.shape}")
logger.info(f"Number of rows (genes): {len(gctoo.data_df)}")
logger.info(f"Number of columns (samples): {len(gctoo.data_df.columns)}")

# Check metadata
if hasattr(gctoo, 'row_metadata_df') and gctoo.row_metadata_df is not None:
    logger.info(f"Row metadata (gene info): {gctoo.row_metadata_df.shape}")
    logger.info(f"Row metadata columns: {gctoo.row_metadata_df.columns.tolist()}")
else:
    logger.warning("Row metadata not found.")

if hasattr(gctoo, 'col_metadata_df') and gctoo.col_metadata_df is not None:
    logger.info(f"Column metadata (sample info): {gctoo.col_metadata_df.shape}")
    logger.info(f"Column metadata columns: {gctoo.col_metadata_df.columns.tolist()}")
else:
    logger.warning("Column metadata not found.")

# Check first few rows of data
logger.info("\nFirst 5 rows and 5 columns of data:")
print(gctoo.data_df.iloc[:5, :5])

## Extracting Gene Information and Identifying Glycogenes

Retrieve gene information and identify glycogene indices.

In [ ]:
# Retrieve gene information
logger.info("Retrieving gene information...")

# Get gene information from gctx file row metadata
if hasattr(gctoo, 'row_metadata_df') and gctoo.row_metadata_df is not None:
    gene_info_df = gctoo.row_metadata_df.copy()
    logger.info(f"Row metadata columns: {gene_info_df.columns.tolist()}")
    
    # Identify gene symbol column
    symbol_col = None
    for col in ['pr_gene_symbol', 'gene_symbol', 'symbol', 'gene_name']:
        if col in gene_info_df.columns:
            symbol_col = col
            break
    
    if symbol_col is None:
        # Index may be gene symbol
        logger.info("Gene symbol not found in columns. Checking index...")
        logger.info(f"Index sample: {gene_info_df.index[:10].tolist()}")
        # Use index as gene symbol
        gene_id_to_symbol = {str(idx): str(idx) for idx in gene_info_df.index}
    else:
        logger.info(f"Gene symbol column: {symbol_col}")
        gene_id_to_symbol = dict(zip(gene_info_df.index.astype(str), gene_info_df[symbol_col].astype(str)))
else:
    # Use row index as gene name
    logger.info("Row metadata not found. Using index as gene name.")
    gene_id_to_symbol = {str(idx): str(idx) for idx in gctoo.data_df.index}

logger.info(f"Gene ID to symbol mapping count: {len(gene_id_to_symbol)}")
logger.info(f"Mapping examples: {list(gene_id_to_symbol.items())[:5]}")

In [ ]:
# Identify glycogene indices
glycogene_indices = []
glycogene_symbols = []

# Create reverse lookup map from symbol to ID
symbol_to_id = {v.upper(): k for k, v in gene_id_to_symbol.items()}

for gene_symbol in glycogenes:
    gene_upper = gene_symbol.upper()
    if gene_upper in symbol_to_id:
        gid = symbol_to_id[gene_upper]
        if gid in gctoo.data_df.index or str(gid) in gctoo.data_df.index.astype(str):
            glycogene_indices.append(gid)
            glycogene_symbols.append(gene_symbol)
    # Check if directly in index
    elif gene_upper in [str(idx).upper() for idx in gctoo.data_df.index]:
        for idx in gctoo.data_df.index:
            if str(idx).upper() == gene_upper:
                glycogene_indices.append(idx)
                glycogene_symbols.append(gene_symbol)
                break

logger.info(f"Glycogenes found in data: {len(glycogene_indices)}/{len(glycogenes)}")

# Check genes not found in data
found_symbols_upper = set([s.upper() for s in glycogene_symbols])
missing_genes = [g for g in glycogenes if g.upper() not in found_symbols_upper]
if missing_genes:
    logger.warning(f"Number of genes not found in data: {len(missing_genes)}")
    logger.warning(f"Examples: {missing_genes[:20]}")

## Extracting Glycogene Data

Extract glycogene expression data from GCTX data.

In [ ]:
# Extract glycogene data
logger.info("Extracting glycogene data...")
glycogene_data = gctoo.data_df.loc[glycogene_indices].copy()

# Convert gene IDs to gene symbols
glycogene_data.index = glycogene_symbols

logger.info(f"Extracted data shape: {glycogene_data.shape}")
logger.info(f"Number of genes: {len(glycogene_data)}")
logger.info(f"Number of samples: {len(glycogene_data.columns)}")

# Transpose to wide format (samples as rows, genes as columns)
glycogene_data_wide = glycogene_data.T

logger.info(f"Transposed data shape: {glycogene_data_wide.shape}")
logger.info(f"Number of samples: {len(glycogene_data_wide)}")
logger.info(f"Number of genes: {len(glycogene_data_wide.columns)}")

## Joining with Metadata

Join glycogene data with sample metadata.

In [ ]:
# Retrieve sample metadata
logger.info("Retrieving sample metadata...")

if hasattr(gctoo, 'col_metadata_df') and gctoo.col_metadata_df is not None:
    col_metadata = gctoo.col_metadata_df.copy()
    logger.info(f"Column metadata shape: {col_metadata.shape}")
    logger.info(f"Columns: {col_metadata.columns.tolist()}")
else:
    logger.warning("Column metadata not found. Using sample IDs only.")
    col_metadata = pd.DataFrame(index=gctoo.data_df.columns)

# Set sample ID (sig_id) as index
glycogene_data_wide.index.name = 'sample_id'
glycogene_data_wide = glycogene_data_wide.reset_index()

# Join with metadata
if len(col_metadata) > 0:
    col_metadata_reset = col_metadata.reset_index()
    col_metadata_reset.columns = ['sample_id'] + [col for col in col_metadata_reset.columns if col != 'index'][1:] if 'index' in col_metadata_reset.columns else col_metadata_reset.columns.tolist()
    
    # Create sample_id column if not present
    if 'sample_id' not in col_metadata_reset.columns:
        col_metadata_reset = col_metadata_reset.rename(columns={col_metadata_reset.columns[0]: 'sample_id'})
    
    df_combined = glycogene_data_wide.merge(
        col_metadata_reset,
        on='sample_id',
        how='left'
    )
else:
    df_combined = glycogene_data_wide

logger.info(f"Data shape after join: {df_combined.shape}")
logger.info(f"Columns: {df_combined.columns[:20].tolist()}...")

## Data Formatting and Column Name Cleanup

Format data and clean up column names for Snowflake table.

In [ ]:
# Standardize column names (convert to uppercase)
df_final = df_combined.copy()
df_final.columns = [col.upper() for col in df_final.columns]

# Move SAMPLE_ID to front
cols = df_final.columns.tolist()
if 'SAMPLE_ID' in cols:
    cols.remove('SAMPLE_ID')
    cols = ['SAMPLE_ID'] + cols
    df_final = df_final[cols]

logger.info(f"Final data shape: {df_final.shape}")
logger.info(f"Number of columns: {len(df_final.columns)}")
logger.info(f"First 20 columns: {df_final.columns[:20].tolist()}")
logger.info(f"\nData sample:")
print(df_final.head())

## Snowflake Connection Configuration

Establish connection to Snowflake database.

In [ ]:
def load_private_key() -> bytes:
    """Load private key (DER format)"""
    key_path = os.path.expanduser('~/.ssh/snowflake_rsa_key.pem')
    with open(key_path, "rb") as key_file:
        private_key = serialization.load_pem_private_key(
            key_file.read(),
            password=None,
            backend=default_backend()
        )
    # Encode in DER format
    private_key_der = private_key.private_bytes(
        encoding=serialization.Encoding.DER,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption()
    )
    return private_key_der


def connect_to_snowflake():
    """Connect to Snowflake"""
    try:
        conn = connect(
            user="KOREEDA",
            account="DUETMBM-LL33279",
            private_key=load_private_key(),
            warehouse="BIOINFORMATICS_XS",
            database="BIOINFORMATICS",
            schema="LINCS",
            role="ACCOUNTADMIN",
        )
        logger.info("Snowflake connection successful")
        return conn
    except Exception as e:
        logger.error(f"Snowflake connection error: {e}")
        raise

# Connect to Snowflake
conn = connect_to_snowflake()

## Creating Snowflake Table and Uploading Data

Create the CTL_GLYCO_GENES_WIDE table and upload data.

In [ ]:
from snowflake.connector.pandas_tools import write_pandas

table_name = "CTL_GLYCO_GENES_WIDE"

# Upload data
logger.info(f"Uploading data to table {table_name}...")
logger.info(f"Number of records: {len(df_final):,}")
logger.info(f"Number of columns: {len(df_final.columns)}")

# Replace NaN values with None
df_upload = df_final.replace({np.nan: None})

# Upload using write_pandas (more efficient)
success, nchunks, nrows, _ = write_pandas(
    conn=conn,
    df=df_upload,
    table_name=table_name,
    database='BIOINFORMATICS',
    schema='LINCS',
    auto_create_table=True,
    overwrite=True
)

logger.info(f"Upload complete: success={success}, chunks={nchunks}, rows={nrows}")

## Verifying Data

Verify the uploaded data.

In [ ]:
cursor = conn.cursor()

# Check table record count
count_query = f"SELECT COUNT(*) FROM BIOINFORMATICS.LINCS.{table_name}"
cursor.execute(count_query)
row_count = cursor.fetchone()[0]
logger.info(f"Number of records in table: {row_count:,}")

# Get sample data
sample_query = f"SELECT * FROM BIOINFORMATICS.LINCS.{table_name} LIMIT 5"
sample_df = pd.read_sql(sample_query, conn)
logger.info(f"\nSample data (first 5 rows):")
logger.info(f"Number of columns: {len(sample_df.columns)}")
print(sample_df)

# Statistics
logger.info(f"\nStatistics:")
logger.info(f"Total records: {row_count:,}")
logger.info(f"Number of glycogenes: {len(glycogene_symbols)}")

cursor.close()
conn.close()
logger.info("\nProcessing complete")